# Eksploracyjna Analiza Danych

Notatnik bada surowe pliki `True.csv` i `Fake.csv` ze zbioru *Fake and Real News Dataset* (Kaggle). Celem jest opis danych przed czyszczeniem oraz uzasadnienie decyzji preprocessingu i parametrów modelu BiLSTM + Attention.

Konwencje: `1 = Real`, `0 = Fake`. Do modelu trafia wyłącznie kolumna `text` - pozostałe kolumny są analizowane jedynie jako kontekst (`title`, `subject`, `date` to potencjalne źródła wycieku, oddzielnie analizowane w `02_leakage.ipynb`).


## 1. Ogólny przegląd

In [ ]:
import re
from collections import Counter
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

pd.options.plotting.backend = "plotly"

LABEL_NAMES = {0: "Fake", 1: "Real"}
LABEL_COLORS = {0: "#EF553B", 1: "#60D39D"}
RANDOM_STATE = 42


In [ ]:
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
true_df = pd.read_csv(f"{path}/True.csv")
fake_df = pd.read_csv(f"{path}/Fake.csv")
true_df["label"] = 1
fake_df["label"] = 0
news_df = pd.concat([true_df, fake_df], ignore_index=True)

print(f"Real: {len(true_df):,}")
print(f"Fake: {len(fake_df):,}")
print(f"Łącznie: {len(news_df):,}")


In [ ]:
news_df.info()

In [ ]:
news_df.describe(include="all")

In [ ]:
nunique = news_df.nunique().rename("unique").to_frame()
nunique["unique_pct"] = (nunique["unique"] / len(news_df) * 100).round(2)
nunique


In [ ]:
counts = news_df["label"].value_counts().sort_index()
balance_df = pd.DataFrame(
    {
        "klasa": [LABEL_NAMES[i] for i in counts.index],
        "liczba": counts.values,
    }
)
fig = px.bar(
    balance_df,
    x="klasa",
    y="liczba",
    color="klasa",
    color_discrete_map={"Fake": LABEL_COLORS[0], "Real": LABEL_COLORS[1]},
    text_auto=True,
    title="Rozkład klas w surowym zbiorze",
)
fig.update_layout(showlegend=False, xaxis_title="Klasa", yaxis_title="Liczba artykułów")
fig.show()

ratio = counts[0] / counts[1]
print(f"Stosunek Fake/Real: {ratio:.3f}")


**Wniosek (1).** Zbiór ma 44 898 rekordów i 5 kolumn; brak wartości `NaN`. Klasy są bliskie zbalansowania (~52% Fake, ~48% Real, stosunek ≈ 1.10). Liczba unikalnych `title` (38 729) i `text` (38 646) sygnalizuje znaczącą liczbę duplikatów - analizowanych w bloku. Resampling ani ważenie klas nie będą potrzebne.


## 2. Duplikaty

In [ ]:
n_total = len(news_df)
n_dup_text = news_df.duplicated(subset=["text"]).sum()
n_dup_title_text = news_df.duplicated(subset=["title", "text"]).sum()
n_empty_text = (news_df["text"].fillna("").str.strip() == "").sum()

summary = pd.DataFrame(
    {
        "miara": [
            "duplikaty (text)",
            "duplikaty (title, text)",
            "puste (text)",
        ],
        "liczba": [n_dup_text, n_dup_title_text, n_empty_text],
        "% zbioru": [
            round(n_dup_text / n_total * 100, 2),
            round(n_dup_title_text / n_total * 100, 2),
            round(n_empty_text / n_total * 100, 2),
        ],
    }
)
summary


In [ ]:
per_class = (
    news_df.groupby("label")
    .apply(lambda g: g.duplicated(subset=["text"]).sum(), include_groups=False)  # type: ignore
    .rename(index=LABEL_NAMES)
    .rename("duplikaty (text)")
    .to_frame()
)
per_class["% klasy"] = (
    per_class["duplikaty (text)"]
    / news_df["label"].map(LABEL_NAMES).value_counts()
    * 100
).round(2)
per_class


In [ ]:
# tekst pojawiający się jednocześnie z etykietą Fake i Real (label noise)
label_cross = news_df.groupby("text")["label"].nunique()
n_cross = (label_cross > 1).sum()
print(f"Teksty o niejednoznacznej etykiecie (Fake i Real): {n_cross}")
if n_cross:
    noisy_texts = label_cross[label_cross > 1].index

noisy_texts

In [ ]:
top_dups = news_df["text"].replace("", np.nan).dropna().value_counts().head(5)
for i, (text, count) in enumerate(top_dups.items(), 1):
    preview = text[:100].replace("\n", " ")  # type: ignore
    print(f"{i}. (liczba przypadkow: {count}): {preview}")


**Wniosek (2).** W zbiorze jest ok. 6 tys. duplikatów `text` (ponad 13%) - przeważają w klasie Fake. Pojawiają się także rekordy z pustym `text` (głównie Fake - krótkie wpisy „YouTube only”). Brak konfliktów etykiet (ten sam tekst raz Real, raz Fake), więc duplikaty można bezpiecznie usunąć w pipeline'ie czyszczenia bez utraty etykiety.


## 3. Analiza metadanych

In [ ]:
subject_ct = pd.crosstab(news_df["subject"], news_df["label"].map(LABEL_NAMES))
subject_ct = subject_ct.reindex(columns=["Fake", "Real"], fill_value=0)
subject_ct["razem"] = subject_ct.sum(axis=1)
subject_ct = subject_ct.sort_values("razem", ascending=False)
subject_ct


In [ ]:
stacked = (
    subject_ct.drop(columns="razem")
    .reset_index()
    .melt(id_vars="subject", var_name="klasa", value_name="liczba")
)
fig = px.bar(
    stacked,
    x="subject",
    y="liczba",
    color="klasa",
    color_discrete_map={"Fake": LABEL_COLORS[0], "Real": LABEL_COLORS[1]},
    title="Rozkład tematów",
)
fig.update_layout(xaxis_title="Subject", yaxis_title="Liczba", barmode="stack")
fig.show()


In [ ]:
news_df["date_parsed"] = pd.to_datetime(news_df["date"], errors="coerce")
n_unparseable = news_df["date_parsed"].isna().sum()
print(f"Nieparsowalne daty: {n_unparseable} ({n_unparseable / len(news_df):.2%})")
print(
    f"Zakres czasowy: {news_df['date_parsed'].min()} – {news_df['date_parsed'].max()}"
)


In [ ]:
monthly = (
    news_df.dropna(subset=["date_parsed"])
    .assign(klasa=lambda d: d["label"].map(LABEL_NAMES))
    .groupby([pd.Grouper(key="date_parsed", freq="ME"), "klasa"])
    .size()
    .reset_index(name="liczba")
)
fig = px.line(
    monthly,
    x="date_parsed",
    y="liczba",
    color="klasa",
    color_discrete_map={"Fake": LABEL_COLORS[0], "Real": LABEL_COLORS[1]},
    title="Liczba artykułów w czasie (miesięcznie)",
)
fig.update_layout(xaxis_title="Miesiąc", yaxis_title="Liczba artykułów")
fig.show()


In [ ]:
title_words = news_df["title"].str.split().str.len()
title_stats = title_words.groupby(news_df["label"].map(LABEL_NAMES)).describe().round(1)
title_stats


**Wniosek (3).** `subject` jest niemal idealnym separatorem klas - kategorie typu `politicsNews` / `worldnews` występują wyłącznie w Real, a `News` / `politics` / `left-news` wyłącznie w Fake. To dowodzi, że kolumna `subject` to czysty wyciek danych i nie może wejść do modelu. Daty z pliku Fake mają niestandardowy format i są w całości nieparsowalne przez domyślny `pd.to_datetime` (~52% wszystkich rekordów); zakres czasowy Real to 2016-01 – 2017-12. Tytuły w Fake są dłuższe i bardziej clickbaitowe (mediana 14 vs 10 słów). Wszystkie te obserwacje uzasadniają dropowanie `subject`, `date`, `title` i operowanie tylko na `text`.

## 4. Długość tekstu

In [ ]:
news_df["char_count"] = news_df["text"].str.len()
news_df["word_count"] = news_df["text"].str.split().str.len()


In [ ]:
stats = (
    news_df.groupby(news_df["label"].map(LABEL_NAMES))[["char_count", "word_count"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(1)
)
stats


In [ ]:
plot_df = news_df[["label", "word_count"]].copy()
plot_df["klasa"] = plot_df["label"].map(LABEL_NAMES)
fig = px.histogram(
    plot_df.query("word_count <= 2000"),
    x="word_count",
    color="klasa",
    nbins=80,
    color_discrete_map={"Fake": LABEL_COLORS[0], "Real": LABEL_COLORS[1]},
    barmode="overlay",
    opacity=0.6,
    title="Rozkład długości artykułu w słowach (przycięte do 2000)",
)
fig.update_layout(xaxis_title="Liczba słów", yaxis_title="Liczba artykułów")
fig.show()


In [ ]:
fig = px.box(
    plot_df,
    x="klasa",
    y="word_count",
    color="klasa",
    color_discrete_map={"Fake": LABEL_COLORS[0], "Real": LABEL_COLORS[1]},
    title="Boxplot długości artykułu w słowach",
    points=False,
    log_y=True,
)
fig.update_layout(showlegend=False, yaxis_title="Liczba słów (skala log)")
fig.show()


In [ ]:
very_short = news_df[news_df["word_count"] < 10]
very_long = news_df[news_df["word_count"] > 3000]
print(f"Artykuły < 10 słów: {len(very_short):,}")
print(
    f"  Fake: {(very_short['label'] == 0).sum():,}, Real: {(very_short['label'] == 1).sum():,}"
)
print(f"Artykuły > 3000 słów: {len(very_long):,}")
print(
    f"  Fake: {(very_long['label'] == 0).sum():,}, Real: {(very_long['label'] == 1).sum():,}"
)

print("\nPrzykłady krótkich artykułów:")
very_short[["text", "word_count", "label"]].head()


**Wniosek (4).** Mediana długości to ~370 słów dla obu klas, ale Fake ma cięższy ogon (99-percentyl > 1500 słów; pojedyncze rekordy > 7000). 95% artykułów mieści się w ok. **900 słowach** - to rozsądny kandydat na `max_seq_len` (BiLSTM bez paddingu do skrajnych outlierów). Krótkich artykułów (<10 słów) jest kilkaset, niemal wszystkie w klasie Fake - często same URL-e do YouTube. **Próg filtracji**: usunąć teksty z `word_count < 10` po czyszczeniu.


## 5. Cechy językowe

In [ ]:
# heurystyka „angielskości": udział liter ASCII + obecność typowych stopwords
import string

ENGLISH_HINTS = {"the", "and", "of", "to", "in", "a", "is", "that", "for", "on"}

sample = news_df.sample(2000, random_state=RANDOM_STATE)
ascii_letters = set(string.ascii_letters)


def ascii_ratio(s: str) -> float:
    if not s:
        return 0.0
    letters = [char for char in s if char.isalpha()]
    if not letters:
        return 0.0
    return sum(char in ascii_letters for char in letters) / len(letters)


def has_english_hints(s: str) -> bool:
    tokens = set(re.findall(r"[a-zA-Z]+", s.lower())[:200])
    return len(tokens & ENGLISH_HINTS) >= 3


sample = sample.assign(
    ascii_ratio=sample["text"].fillna("").map(ascii_ratio),
    english_hints=sample["text"].fillna("").map(has_english_hints),
)
non_eng = sample[(sample["ascii_ratio"] < 0.95) | (~sample["english_hints"])]
print(f"Próbka: {len(sample):,} artykułów")
print(
    f"Podejrzanych nie-angielskich: {len(non_eng):,} ({len(non_eng) / len(sample):.2%})"
)


In [ ]:
# udziały specjalnych klas znaków
def char_ratio(text: str, predicate) -> float:
    if not text:
        return 0.0
    return sum(1 for char in text if predicate(char)) / len(text)


feat = pd.DataFrame(
    {
        "klasa": news_df["label"].map(LABEL_NAMES),
        "upper_ratio": news_df["text"]
        .fillna("")
        .map(lambda t: char_ratio(t, str.isupper)),
        "digit_ratio": news_df["text"]
        .fillna("")
        .map(lambda t: char_ratio(t, str.isdigit)),
        "punct_ratio": news_df["text"]
        .fillna("")
        .map(lambda t: char_ratio(t, lambda c: c in string.punctuation)),
        "allcaps_words": news_df["text"].fillna("").str.count(r"\b[A-Z]{3,}\b"),
    }
)
feat.groupby("klasa").mean().round(4)


Długość i liczba zdań mogą być sygnałem stylistycznym odróżniającym klasy.                                                    
  `sentence_count` - liczba zdań (zliczone wystąpienia `.`, `!`, `?`);
  `avg_sentence_len` -
 średnia liczba słów na zdanie.

In [ ]:
news_df["sentence_count"] = news_df["text"].fillna("").str.count(r"[.!?]+")
news_df["avg_sentence_len"] = news_df["word_count"] / news_df["sentence_count"].replace(
    0, np.nan
)
(
    news_df.groupby(news_df["label"].map(LABEL_NAMES))[
        ["sentence_count", "avg_sentence_len"]
    ]
    .describe(percentiles=[0.5, 0.95])
    .round(2)
)


In [ ]:
patterns = pd.DataFrame(
    {
        "klasa": news_df["label"].map(LABEL_NAMES),
        "has_url": news_df["text"].fillna("").str.contains(r"https?://", regex=True),
        "has_handle": news_df["text"].fillna("").str.contains(r"@\w+", regex=True),
        "has_hashtag": news_df["text"].fillna("").str.contains(r"#\w+", regex=True),
    }
)
(patterns.groupby("klasa").mean() * 100).round(2)


**Wniosek (5).** Zbiór jest zasadniczo angielski (>99% próbki), więc detekcja języka nie jest konieczna w pipelinie. Fake ma kilkukrotnie więcej słów ALL-CAPS, znaków `!`/`?` i URL-i / Twitterowych handli - to jednocześnie sygnał stylu i potencjalny wyciek (analiza w `02_leakage.ipynb`). Liczba zdań jest porównywalna, ale rozkład w Fake ma cięższe ogony, spójnie z wcześniejszymi obserwacjami długości.


## 6. Słownictwo i pokrycie GloVe 300d

In [ ]:
import nltk

nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("english"))

TOKEN_RE = re.compile(r"[a-zA-Z]+")


def tokenize(text: str) -> list[str]:
    return [t.lower() for t in TOKEN_RE.findall(text or "")]


def freq_for(label: int) -> Counter:
    counter: Counter[str] = Counter()
    for txt in news_df.loc[news_df["label"] == label, "text"].fillna(""):
        counter.update(tokenize(txt))
    return counter


real_freq = freq_for(1)
fake_freq = freq_for(0)
vocab = set(real_freq) | set(fake_freq)
print(f"Tokenów Real: {sum(real_freq.values()):,}")
print(f"Tokenów Fake: {sum(fake_freq.values()):,}")
print(f"Słownictwo łączne: {len(vocab):,}")


In [ ]:
all_freq = real_freq + fake_freq
freqs = np.array(sorted(all_freq.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
fig = px.scatter(
    x=ranks,
    y=freqs,
    log_x=True,
    log_y=True,
    title="Prawo Zipfa - częstość vs ranga (log-log)",
    labels={"x": "Ranga", "y": "Częstość"},
)
fig.update_traces(marker=dict(size=3, opacity=0.6))
fig.show()


In [ ]:
def top_no_stop(counter: Counter, k: int = 30) -> pd.DataFrame:
    items = [(w, c) for w, c in counter.most_common() if w not in STOPWORDS][:k]
    return pd.DataFrame(items, columns=["słowo", "częstość"])


real_top = top_no_stop(real_freq).assign(klasa="Real")
fake_top = top_no_stop(fake_freq).assign(klasa="Fake")
top_df = pd.concat([real_top, fake_top])

fig = make_subplots(rows=1, cols=2, subplot_titles=["Real", "Fake"])

fig.add_trace(
    go.Bar(
        x=real_top["częstość"],
        y=real_top["słowo"],
        orientation="h",
        marker_color=LABEL_COLORS[1],
        name="Real",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=fake_top["częstość"],
        y=fake_top["słowo"],
        orientation="h",
        marker_color=LABEL_COLORS[0],
        name="Fake",
    ),
    row=1,
    col=2,
)

fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_layout(
    showlegend=False,
    height=700,
    title="Top 30 słów per klasa (po usunięciu stopwords)",
)
fig.show()


In [ ]:
GLOVE_PATH = Path.cwd() / "data" / "glove" / "glove_2024_wikigiga_300d.txt"


def normalize(token: str) -> str:
    return token.lower().strip(string.punctuation)


if GLOVE_PATH.exists():
    glove_vocab = set()
    with GLOVE_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            glove_vocab.add(line.split(" ", 1)[0])
    print(f"GloVe 6B 300d vocab: {len(glove_vocab):,}")

    raw_oov = {w for w in vocab if w not in glove_vocab}
    norm_oov = {w for w in vocab if normalize(w) not in glove_vocab}
    raw_token_oov = sum(c for w, c in all_freq.items() if w not in glove_vocab)
    norm_token_oov = sum(
        c for w, c in all_freq.items() if normalize(w) not in glove_vocab
    )
    total_tokens = sum(all_freq.values())

    coverage = pd.DataFrame(
        {
            "wariant": ["surowe (lower+alpha)", "po normalizacji"],
            "OOV typów %": [
                round(len(raw_oov) / len(vocab) * 100, 2),
                round(len(norm_oov) / len(vocab) * 100, 2),
            ],
            "OOV tokenów %": [
                round(raw_token_oov / total_tokens * 100, 2),
                round(norm_token_oov / total_tokens * 100, 2),
            ],
        }
    )
    display(coverage)

    sample_oov = sorted(raw_oov, key=lambda w: all_freq[w], reverse=True)[:20]
    print("\nNajczęstsze OOV (przed normalizacją):")
    print(sample_oov)
else:
    print(f"Brak pliku GloVe: {GLOVE_PATH}")
    print(
        "Pobierz `glove.6B.300d.txt` z https://nlp.stanford.edu/projects/glove/ "
        "i umieść w data/glove/, następnie ponów tę komórkę."
    )


**Wniosek (6).** Słownictwo zawiera ok. 100 tys. typów; rozkład spełnia prawo Zipfa (proste log-log) - bez anomalii. Top słowa Real są zdominowane przez referencje do agencji, geografii i polityki USA; Fake są bardziej emocjonalne. **Przed treningiem** sprawdzić pokrycie GloVe 300d na słownictwie *po* czyszczeniu - jeśli OOV tokenów > 5%, rozważyć agresywniejszą normalizację (lower-case, strip punctuation, mapowanie cyfr na placeholder).


## 7. Podsumowanie i decyzje

Decyzje wynikające z EDA, które przechodzą do `03_cleaning.ipynb` i dalszych etapów:

**Kolumny.** Dropujemy `title`, `subject`, `date` - `subject` jest oczywistym leakage'em (rozłączne kategorie per klasa), `date` ma niejednolity format i nie wnosi treści, `title` jest stylistycznym sygnałem skorelowanym z klasą. Do modelu trafia wyłącznie `text` + `label`.

**Filtracja wierszy.** Usunąć duplikaty `text` (≈ 6 tys.), puste / krótkie teksty (`word_count < 10` po czyszczeniu).

**Sekwencja modelu.** `max_seq_len ≈ 900` pokrywa 95% artykułów; alternatywnie 700 (≈ 90%) dla mniejszego batcha.

**Sygnały leakage do dalszej analizy** (`02_leakage.ipynb`):
- prefix Reuters (`(Reuters)`, dateline `WASHINGTON (Reuters) -`),
- nadreprezentacja URL-i, handli `@…`, hashtagów w Fake,
- ALL-CAPS jako stylistyczny marker Fake.

**GloVe.** Po cleaningu zweryfikować OOV - jeśli > 5%, dodać normalizację (lower-case, strip-punctuation, mapowanie liczb).
